# 06 — Explainability & Sensitivity Analysis

This notebook provides:
1. SHAP analysis on the winning ETA model
2. Reliability Score sensitivity analysis
3. Feature importance interpretation

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dabba.config import get_config

config = get_config()

## 1. SHAP Analysis on Winning ETA Model

In [ ]:
# Load the saved model and data
import joblib
from pathlib import Path

model_path = config.best_eta_model_path
if Path(model_path).exists():
    model = joblib.load(model_path)
    print(f'Loaded ETA model from {model_path}')
else:
    print(f'⚠️ Model not found at {model_path}. Run `make train` first.')

In [ ]:
try:
    import shap
    
    # Get feature names from preprocessor
    preprocessor = model.named_steps['preprocessor']
    reg = model.named_steps['model']
    
    # Get feature names
    num_features = preprocessor.transformers_[0][2] if hasattr(preprocessor.transformers_[0][2], '__iter__') else []
    cat_encoder = preprocessor.transformers_[1][1]
    cat_features = list(cat_encoder.get_feature_names_out()) if hasattr(cat_encoder, 'get_feature_names_out') else []
    feature_names = list(num_features) + cat_features
    
    print(f'Feature names: {feature_names}')
    
except ImportError:
    print('⚠️ SHAP not installed. Install with: pip install shap')

In [ ]:
# SHAP summary plot (global feature importance)
try:
    import shap
    from dabba.data.loaders import load_delivery
    from dabba.data.cleaning import clean_delivery
    from dabba.features.delivery_features import add_delivery_features
    
    df = load_delivery(config)
    df = clean_delivery(df, config)
    df = add_delivery_features(df, config)
    
    feature_cols = [c for c in df.columns if c in [
        'haversine_distance_km', 'traffic_ordinal', 'is_festival',
        'delivery_person_age', 'delivery_person_ratings', 'vehicle_condition',
    ]]
    feature_cols += [c for c in df.columns if c.startswith('order_hour_bucket_')]
    
    X = df[feature_cols].fillna(0).sample(n=min(500, len(df)), random_state=42)
    
    # Create SHAP explainer
    explainer = shap.TreeExplainer(reg) if hasattr(reg, 'feature_importances_') else shap.KernelExplainer(model.predict, shap.sample(X, 100))
    shap_values = explainer.shap_values(X)
    
    # Summary plot
    shap.summary_plot(shap_values, X, feature_names=feature_names, show=False)
    plt.title('SHAP Summary — ETA Model Feature Importance')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f'SHAP analysis requires the trained model and data: {e}')

## 2. SHAP Force Plots (Individual Predictions)

In [ ]:
# Show SHAP force plots for 3 example orders (fast/normal/slow)
try:
    import shap
    
    # Get predictions to find fast/normal/slow examples
    predictions = model.predict(X)
    
    fast_idx = np.argmin(predictions)
    slow_idx = np.argmax(predictions)
    median_idx = np.argsort(predictions)[len(predictions) // 2]
    
    print(f'Fast order: predicted {predictions[fast_idx]:.1f} min')
    print(f'Normal order: predicted {predictions[median_idx]:.1f} min')
    print(f'Slow order: predicted {predictions[slow_idx]:.1f} min')
    
    for label, idx in [('Fast', fast_idx), ('Normal', median_idx), ('Slow', slow_idx)]:
        print(f'\n--- {label} Order ---')
        shap.force_plot(
            explainer.expected_value, shap_values[idx],
            X.iloc[idx], feature_names=feature_names,
            matplotlib=True, show=False
        )
        plt.title(f'SHAP Force Plot — {label} Delivery')
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f'Force plot requires SHAP: {e}')

## 3. Reliability Score Sensitivity Analysis

Show how restaurant rankings shift as weights change.

In [ ]:
from dabba.evaluation.business_cost import compute_reliability_score

# Simulated restaurant data
np.random.seed(42)
n_restaurants = 20
ratings = np.random.uniform(2.5, 5.0, n_restaurants)
sentiments = np.random.uniform(-0.5, 1.0, n_restaurants)
delay_risks = np.random.uniform(0.1, 0.9, n_restaurants)

# Test different weight configurations
weight_configs = [
    {'w_rating': 0.4, 'w_sentiment': 0.3, 'w_delay': 0.3, 'label': 'Default (0.4/0.3/0.3)'},
    {'w_rating': 0.6, 'w_sentiment': 0.2, 'w_delay': 0.2, 'label': 'Rating-heavy (0.6/0.2/0.2)'},
    {'w_rating': 0.2, 'w_sentiment': 0.5, 'w_delay': 0.3, 'label': 'Sentiment-heavy (0.2/0.5/0.3)'},
    {'w_rating': 0.3, 'w_sentiment': 0.3, 'w_delay': 0.4, 'label': 'Delay-penalizing (0.3/0.3/0.4)'},
]

print('=== Sensitivity Analysis: How Weights Affect Rankings ===\n')
for cfg in weight_configs:
    # Temporarily override config
    original = (config.reliability_w_rating, config.reliability_w_sentiment, config.reliability_w_delay)
    config.reliability_w_rating = cfg['w_rating']
    config.reliability_w_sentiment = cfg['w_sentiment']
    config.reliability_w_delay = cfg['w_delay']
    
    scores = compute_reliability_score(ratings, sentiments, delay_risks, config)
    top3 = np.argsort(scores)[-3:][::-1]
    
    print(f"{cfg['label']}:")
    for i, idx in enumerate(top3):
        print(f"  #{i+1}: Restaurant {idx} (score={scores[idx]:.3f}, rating={ratings[idx]:.2f}, sent={sentiments[idx]:.2f}, delay={delay_risks[idx]:.2f})")
    print()
    
    # Restore
    config.reliability_w_rating = original[0]
    config.reliability_w_sentiment = original[1]
    config.reliability_w_delay = original[2]

In [ ]:
# Visualize ranking shifts
ranking_data = []
for cfg in weight_configs:
    config.reliability_w_rating = cfg['w_rating']
    config.reliability_w_sentiment = cfg['w_sentiment']
    config.reliability_w_delay = cfg['w_delay']
    
    scores = compute_reliability_score(ratings, sentiments, delay_risks, config)
    ranks = np.argsort(np.argsort(-scores))  # Rank 0 = best
    
    for i in range(n_restaurants):
        ranking_data.append({
            'restaurant': i,
            'weight_config': cfg['label'],
            'rank': ranks[i],
            'score': scores[i]
        })
    
    config.reliability_w_rating = 0.4
    config.reliability_w_sentiment = 0.3
    config.reliability_w_delay = 0.3

ranking_df = pd.DataFrame(ranking_data)

fig, ax = plt.subplots(figsize=(12, 6))
for rest_id in range(5):  # Show top 5 restaurants
    subset = ranking_df[ranking_df['restaurant'] == rest_id]
    ax.plot(subset['weight_config'], subset['rank'], marker='o', label=f'Restaurant {rest_id}')

ax.set_ylabel('Rank (lower = better)')
ax.set_title('Ranking Sensitivity: How Restaurant Rankings Shift with Weight Changes')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Key Takeaways

- SHAP reveals which features drive delivery time predictions
- The force plots show why specific orders are fast/slow
- The reliability score is sensitive to weight choices — the sensitivity analysis helps justify the defaults